# <center> <font color="#004C46">Maestria en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Institucion](https://img.shields.io/badge/INSTITUCI%C3%93N-TECNOL%C3%93GICO_DE_MONTERREY-003A70?style=for-the-badge)](https://tec.mx)
[![Programa](https://img.shields.io/badge/PROGRAMA-MNA-004C46?style=for-the-badge)](https://tec.mx)
[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-1255C?style=for-the-badge)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-F37626?style=flat-square&logo=jupyter&logoColor=white)](https://jupyter.org/)
[![Prophet](https://img.shields.io/badge/Prophet-3B5998?style=flat-square&logo=meta&logoColor=white)](https://facebook.github.io/prophet/)
[![DeepAR](https://img.shields.io/badge/DeepAR-FF9900?style=flat-square&logo=amazonaws&logoColor=white)](https://ts.gluon.ai/)
[![XGBoost](https://img.shields.io/badge/XGBoost-017A3A?style=flat-square)](https://xgboost.readthedocs.io/)
[![LightGBM](https://img.shields.io/badge/LightGBM-9ACD32?style=flat-square)](https://lightgbm.readthedocs.io/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/IntegradorIMSS2026Team01/EpiForecast-MX)

</center>

---

## **<font color="#004C46">Semana 7: Avance 5 -- Modelo Final (Ensemble Learning)</font>**

### **<font color="#7B2132">Proyecto: EpiForecast-MX</font>**

**Materia:** Proyecto Integrador -- TC5035.10

> **Titulo:** Generalizacion de modelos nacionales de pronostico epidemiologico hacia un enfoque modular con desagregacion por sexo y entidad federativa en Mexico.

---

### **Objetivos de esta entrega**

**Objetivo 3.5** -- Implementar al menos 4 modelos de ensemble learning (homogeneo y heterogeneo) utilizando los mejores modelos individuales como base para stacking/blending.

**Objetivo 3.6** -- Comparar y evaluar todos los modelos entrenados con una tabla comparativa ordenada por la metrica principal (**SMAPE**), acompanada de al menos 2 metricas adicionales (MASE, RMSE), tiempos de entrenamiento y argumentos solidos para la seleccion del modelo final.

---

### **Cuerpo Docente y Patrocinio**

* **Dra. Grettel Barcelo Alonso** | Profesora Titular y Asesora
* **Dra. Ruth Perez-Hernandez** | Lider del Proyecto IMSS
* **Dra. Lina Diaz Castro** | Investigadora en Psiquiatria IMSS

---

## **<center> <font color="#004C46">Equipo de Desarrollo #1</font> </center>**

<table style="width:100%; border:none; border-collapse:collapse;">
<tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
<img src="https://proyectointegrador.org/JARS20206.jpg" width="140px" style="border-radius:10px;">
<br><h4>Javier Augusto Rebull Saucedo</h4><code>A01795838</code><br>
<small>MNA Student &middot; Sr. Associate Dev -- Santander</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
<img src="https://proyectointegrador.org/JARCOS2026.png" width="140px" style="border-radius:10px;">
<br><h4>Juan Carlos Perez Nava</h4><code>A01795941</code><br>
<small>MNA Student &middot; IT Professional -- IMSS</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
<img src="https://proyectointegrador.org/LUIS2026.jpg" width="140px" style="border-radius:10px;">
<br><h4>Luis Gerardo Sanchez Salazar</h4><code>A01232963</code><br>
<small>MNA Student &middot; Sr. Controls Engineer -- Tesla</small>
    </td>
</tr>
</table>

**Fecha de entrega:** Marzo 2026

---

In [ ]:
# Cell 1 -- Imports, paleta IMSS y configuracion SOLID
from __future__ import annotations

from dataclasses import dataclass, field
import os
from pathlib import Path
import pickle
import shutil
import warnings

from IPython.display import Image, display
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch, Patch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

os.chdir(Path("..").resolve())


# ── Paleta IMSS institucional 2026 ──────────────────────────────
@dataclass(frozen=True)
class IMSSPalette:
    """Colores institucionales del IMSS 2026."""

    verde: str = "#004C46"
    verde_claro: str = "#00796B"
    rojo: str = "#7B2132"
    dorado: str = "#DAA520"
    gris_oscuro: str = "#2C2C2C"
    gris_claro: str = "#F5F5F5"
    blanco: str = "#FFFFFF"

    prophet: str = "#004D40"  # teal institucional
    deepar: str = "#880E4F"  # vino institucional
    ensemble: str = "#FF6F00"  # naranja IMSS
    stacking: str = "#1A237E"  # indigo profundo

    @property
    def modelo(self) -> dict[str, str]:
        return {
            "Prophet": self.prophet,
            "DeepAR": self.deepar,
            "Ensemble": self.ensemble,
            "Stacking": self.stacking,
        }

    @property
    def padecimiento(self) -> dict[str, str]:
        return {
            "Alzheimer": "#1A237E",
            "Depresion": "#E65100",
            "Parkinson": "#004D40",
        }


PAL = IMSSPalette()


# ── Configuracion del notebook ──────────────────────────────────
@dataclass(frozen=True)
class NotebookConfig:
    """Rutas y constantes del proyecto."""

    repo: Path = Path()
    fig_dir: Path = Path("notebooks/figuras_avance5")

    padecimientos: tuple[str, ...] = ("Alzheimer", "Depresion", "Parkinson")
    pad_display: dict[str, str] = field(
        default_factory=lambda: {
            "Alzheimer": "Alzheimer (G30)",
            "Depresion": "Depresion (F32)",
            "Parkinson": "Parkinson (G20)",
        }
    )
    # Nombre del directorio overlay (sin tilde)
    pad_overlay: dict[str, str] = field(
        default_factory=lambda: {
            "Alzheimer": "Alzheimer",
            "Depresion": "Depresion",
            "Parkinson": "Parkinson",
        }
    )
    # Prefijo en el nombre del archivo overlay (CON tilde para Dep)
    pad_file_prefix: dict[str, str] = field(
        default_factory=lambda: {
            "Alzheimer": "Alzheimer",
            "Depresion": "Depresi\u00f3n",
            "Parkinson": "Parkinson",
        }
    )
    modelos: tuple[str, ...] = ("Prophet", "DeepAR", "Ensemble", "Stacking")
    model_prefixes: dict[str, str] = field(
        default_factory=lambda: {
            "Prophet": "Prophet",
            "DeepAR": "Deepar",
            "Ensemble": "Ensemble",
            "Stacking": "Stacking",
        }
    )

    @property
    def excel_metricas(self) -> Path:
        return self.repo / "reports/forecasts/comparacion_modelos/comparacion_metricas.xlsx"

    @property
    def excel_produccion(self) -> Path:
        return self.repo / "reports/ProdDetails/tabla_333_modelos_produccion.xlsx"

    @property
    def models_dir(self) -> Path:
        return self.repo / "models"

    @property
    def forecasts_dir(self) -> Path:
        return self.repo / "reports/forecasts"

    @property
    def overlay_base(self) -> Path:
        return self.repo / "reports/forecasts/comparacion_modelos"


cfg = NotebookConfig()
cfg.fig_dir.mkdir(parents=True, exist_ok=True)

# ── Seaborn theme IMSS ──────────────────────────────────────────
sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 150,
        "savefig.dpi": 200,
        "font.family": "sans-serif",
        "axes.edgecolor": PAL.verde,
        "axes.labelcolor": PAL.gris_oscuro,
        "text.color": PAL.gris_oscuro,
    },
)

SEXO_MAP = {
    "incrementos_hombres": "Hombres",
    "incrementos_mujeres": "Mujeres",
    "incrementos_total": "General",
    "hombres": "Hombres",
    "mujeres": "Mujeres",
    "general": "General",
}


# ── Helpers ─────────────────────────────────────────────────────
def guardar(fig: plt.Figure, nombre: str) -> None:
    path = cfg.fig_dir / f"{nombre}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"  Guardado: {path}")


def _imss_table_style() -> list[dict]:
    """Estilo de tabla institucional IMSS."""
    return [
        {
            "selector": "th",
            "props": [
                ("background-color", PAL.verde),
                ("color", PAL.blanco),
                ("font-weight", "bold"),
                ("border", "1px solid #ddd"),
                ("padding", "6px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #ddd"),
                ("padding", "5px"),
                ("font-size", "11px"),
            ],
        },
    ]


def _overlay_path(pad: str, entidad: str, sexo: str) -> str:
    """Construye ruta al PNG overlay de comparacion de modelos."""
    overlay_dir = cfg.overlay_base / cfg.pad_overlay[pad] / "overlay"
    if not overlay_dir.exists():
        return ""
    prefix = cfg.pad_file_prefix[pad]
    sexo_key = SEXO_MAP.get(sexo, sexo).lower()
    ent_clean = entidad.replace(" ", "_")
    # Busqueda directa
    candidate = overlay_dir / f"{prefix}_{ent_clean}_{sexo_key}.png"
    if candidate.exists():
        return str(candidate)
    # Busqueda fuzzy
    for f in overlay_dir.iterdir():
        if ent_clean.lower() in f.stem.lower() and sexo_key in f.stem.lower():
            return str(f)
    return ""


print(f"Configuracion SOLID cargada. Repo: {cfg.repo.resolve()}")
print(f"CWD: {Path.cwd()}")
print(f"Paleta IMSS: {PAL.modelo}")

In [ ]:
# Cell 2 -- MetricsLoader (Single Responsibility)
class MetricsLoader:
    """Carga centralizada de todas las fuentes de datos."""

    def __init__(self, config: NotebookConfig) -> None:
        self.cfg = config
        self.resumen: pd.DataFrame = pd.DataFrame()
        self.detalle: pd.DataFrame = pd.DataFrame()
        self.completos: dict[str, pd.DataFrame] = {}
        self.forecasts_nac: dict[str, dict[str, pd.DataFrame]] = {}
        self.produccion: pd.DataFrame = pd.DataFrame()
        self.detalle_semanal: pd.DataFrame = pd.DataFrame()
        self.yamls: dict[str, dict] = {}

    def load_all(self) -> MetricsLoader:
        self._load_excel_metricas()
        self._load_produccion()
        self._load_completos()
        self._load_forecasts()
        self._load_yamls()
        self._print_summary()
        return self

    # ── Privados ────────────────────────────────────────────────
    def _load_excel_metricas(self) -> None:
        xl = self.cfg.excel_metricas
        assert xl.exists(), f"No existe: {xl}"
        self.resumen = pd.read_excel(xl, sheet_name="Resumen")
        self.detalle = pd.read_excel(xl, sheet_name="Detalle")
        print(f"Excel metricas: Resumen {self.resumen.shape}, Detalle {self.detalle.shape}")

    def _load_produccion(self) -> None:
        xl = self.cfg.excel_produccion
        if not xl.exists():
            print(f"  AVISO: no existe {xl}")
            return
        try:
            self.produccion = pd.read_excel(xl, sheet_name="Producci\u00f3n")
        except ValueError:
            self.produccion = pd.read_excel(xl, sheet_name=0)
        self.detalle_semanal = pd.read_excel(xl, sheet_name="Detalle Semanal")
        # Normalizar padecimiento sin tilde para joins
        self.produccion["pad_key"] = self.produccion["padecimiento"].str.replace(
            "\u00f3", "o", regex=False
        )
        print(f"Tabla produccion: {self.produccion.shape[0]} filas")

    def _load_completos(self) -> None:
        for modelo in self.cfg.modelos:
            prefix = self.cfg.model_prefixes[modelo]
            for pad in self.cfg.padecimientos:
                key = f"{modelo}_{pad}"
                path = self.cfg.models_dir / modelo.lower() / pad / f"{prefix}_{pad}_completo.csv"
                if path.exists():
                    self.completos[key] = pd.read_csv(path)
                else:
                    print(f"  AVISO: no existe {path.name}")
        print(f"Completos cargados: {len(self.completos)} de 12")

    def _load_forecasts(self) -> None:
        for modelo in self.cfg.modelos:
            csv_path = (
                self.cfg.forecasts_dir / modelo.lower() / f"all_forecast_{modelo.lower()}.csv"
            )
            if not csv_path.exists():
                continue
            df = pd.read_csv(csv_path, low_memory=False)
            df["ds"] = pd.to_datetime(df["ds"])
            self.forecasts_nac[modelo] = {}
            for pad in self.cfg.padecimientos:
                mask = (
                    df["meta_padecimiento"].str.contains(pad[:5], case=False, na=False)
                    & (df["meta_entidad"] == "Nacional")
                    & (df["meta_modo"] == "general")
                )
                sub = df[mask].copy()
                if not sub.empty:
                    self.forecasts_nac[modelo][pad] = sub
        print(f"Forecasts nacionales: {len(self.forecasts_nac)} modelos")

    def _load_yamls(self) -> None:
        for name in ["prophet", "deepar", "ensemble", "stacking"]:
            path = self.cfg.repo / f"config/models/{name}.yaml"
            if path.exists():
                with path.open() as f:
                    self.yamls[name] = yaml.safe_load(f)
        print(f"YAMLs cargados: {len(self.yamls)}")

    # ── Publicos ────────────────────────────────────────────────
    def load_serie_real(self, pad: str) -> pd.DataFrame:
        for model_key in ["prophet", "ensemble", "stacking", "deepar"]:
            prefix = self.cfg.model_prefixes[
                {
                    "prophet": "Prophet",
                    "ensemble": "Ensemble",
                    "stacking": "Stacking",
                    "deepar": "DeepAR",
                }[model_key]
            ]
            path = self.cfg.models_dir / model_key / pad / f"{prefix}_{pad}_general.csv"
            if path.exists():
                df = pd.read_csv(path)
                df["ds"] = pd.to_datetime(df["ds"])
                return df
        msg = f"No se encontro serie real para {pad}"
        raise FileNotFoundError(msg)

    def _print_summary(self) -> None:
        n_fc = sum(len(v) for v in self.forecasts_nac.values())
        print(
            f"\n--- Resumen de carga ---\n"
            f"  Detalle: {self.detalle.shape[0]} filas"
            f" | Completos: {len(self.completos)}"
            f" | Forecasts: {n_fc} series"
            f" | Produccion: {self.produccion.shape[0]} filas"
        )


loader = MetricsLoader(cfg).load_all()

---

## **<font color="#004C46">ACT 1 -- Marco Teorico: Ensemble Learning en Series de Tiempo Epidemiologicas</font>**

El aprendizaje por ensamble (*ensemble learning*) combina las predicciones de multiples modelos base para producir un pronostico mas robusto y generalizable que cualquier modelo individual. En el contexto de series de tiempo epidemiologicas, donde la incidencia semanal presenta estacionalidad anual, cambios de regimen post-COVID y alta variabilidad entre entidades federativas, la diversificacion de algoritmos resulta especialmente valiosa.

EpiForecast-MX implementa cuatro motores de pronostico:

| Motor | Categoria | Estrategia | Modelos base | Meta-learner |
|:------|:----------|:-----------|:-------------|:-------------|
| **Prophet** | Individual (baseline) | Modelo aditivo bayesiano con estacionalidad Fourier | -- | -- |
| **DeepAR** | Homogeneo (multi-serie) | LSTM autoregresiva probabilistica, 32 series simultaneas | -- | -- |
| **Ensemble** | Heterogeneo (promedio ponderado) | Prophet + XGBoost con 15 features; pesos OOF via Ridge | Prophet + XGBoost | Ridge (no negativo) |
| **Stacking** | Heterogeneo (stacking generalizado) | 3 expertos independientes; ElasticNet como meta-learner | Prophet + ETS + LightGBM | ElasticNet |

### Metricas de evaluacion

La metrica **primaria** de seleccion es **SMAPE** (Symmetric Mean Absolute Percentage Error), que ofrece una interpretacion intuitiva como porcentaje de error y es simetrica ante sub/sobreestimacion. En caso de empate (diferencia < 5%), se utiliza **MASE** como desempate, ya que no se infla con series de baja incidencia. **RMSE** sirve como segundo desempate para capturar errores de gran magnitud.

*Referencias: Singh (2023), VanderPlas (2022), Hyndman & Koehler (2006), Makridakis (1993).*

In [ ]:
# Cell 4 -- Arquitectura de 333 modelos por motor
data_333 = {
    "Nivel": [
        "Estatal (32 entidades)",
        "Nacional",
        "Regional (4 regiones INEGI)",
        "Subtotal por padecimiento",
        "TOTAL por motor (3 pad.)",
        "TOTAL proyecto (4 motores)",
    ],
    "Sexos": ["x 3 (H, M, Comb.)", "x 3", "x 3", "", "", ""],
    "Series": ["96", "3", "12", "111", "333", "1,332 modelos"],
    "Artefactos": [
        "96 .pkl + 96 .csv",
        "3 .pkl + 3 .csv",
        "12 .pkl + 12 .csv",
        "111 .pkl + 112 .csv",
        "333 .pkl + ~339 .csv",
        "2,685 archivos",
    ],
}
df_333 = pd.DataFrame(data_333)
display(
    df_333.style.set_properties(**{"text-align": "left"})
    .set_table_styles(_imss_table_style())
    .hide(axis="index")
    .set_caption("Composicion de los 333 modelos por motor de pronostico")
)

In [ ]:
# Cell 5 -- Fichas tecnicas con datos reales de pkl y Excel
tiempos: dict[str, float] = {}
for modelo in cfg.modelos:
    col = f"Tiempo (s) {modelo}"
    tiempos[modelo] = (
        loader.detalle[col].sum() / 60 if col in loader.detalle.columns else float("nan")
    )


def _load_pkl_params(modelo: str, pad: str = "Depresion") -> dict:
    prefix = cfg.model_prefixes[modelo]
    path = cfg.models_dir / modelo.lower() / pad / f"{prefix}_{pad}_general.pkl"
    if not path.exists():
        return {}
    try:
        with path.open("rb") as f:
            data = pickle.load(f)
        return data.get("params", {})
    except Exception:
        return {}


fichas = []
for modelo in cfg.modelos:
    p = _load_pkl_params(modelo)
    t = f"{tiempos[modelo]:.1f} min" if not np.isnan(tiempos[modelo]) else "N/D"
    if modelo == "Prophet":
        hp = p.get("prophet", p)
        fichas.append(
            {
                "Motor": modelo,
                "Categoria": "Individual (baseline)",
                "Base": "Aditivo bayesiano",
                "Meta-learner": "--",
                "HP clave": f"cps={hp.get('changepoint_prior_scale', '?')}, sp={hp.get('seasonality_prior_scale', '?')}",
                "Tiempo total": t,
            }
        )
    elif modelo == "DeepAR":
        fichas.append(
            {
                "Motor": modelo,
                "Categoria": "Homogeneo (multi-serie)",
                "Base": "LSTM autoregresiva (GluonTS+PyTorch)",
                "Meta-learner": "--",
                "HP clave": "context=104, horizon=52, epochs=300",
                "Tiempo total": t,
            }
        )
    elif modelo == "Ensemble":
        xgb_hp = p.get("xgboost", {})
        fichas.append(
            {
                "Motor": "Ensemble Paralelo",
                "Categoria": "Heterogeneo (prom. ponderado)",
                "Base": "Prophet + XGBoost (15 features)",
                "Meta-learner": "Ridge (OOF, no negativo)",
                "HP clave": f"n_est={xgb_hp.get('n_estimators', '?')}, depth={xgb_hp.get('max_depth', '?')}, lr={xgb_hp.get('learning_rate', '?')}",
                "Tiempo total": t,
            }
        )
    else:
        fichas.append(
            {
                "Motor": "Stacking Generalizado",
                "Categoria": "Heterogeneo (stacking)",
                "Base": "Prophet + ETS + LightGBM",
                "Meta-learner": f"ElasticNet (alpha={p.get('alpha', 0.1)})",
                "HP clave": f"horizon={p.get('horizon', 52)} sem, folds=4",
                "Tiempo total": t,
            }
        )

df_fichas = pd.DataFrame(fichas).set_index("Motor").T
display(
    df_fichas.style.set_properties(**{"text-align": "left", "font-size": "11px"})
    .set_table_styles(_imss_table_style())
    .set_caption("Tabla 1. Fichas tecnicas de los 4 motores de pronostico")
)

---

## **<font color="#004C46">ACT 2 -- Optimizacion de Hiperparametros</font>**

Cada motor realiza busqueda en grilla (*grid search*) con validacion cruzada temporal y pesos progresivos por fold (cv_weights: [0.5, 0.75, 1.0, 1.25]). La metrica de seleccion en CV es **SMAPE ponderado**, con early stopping por timeout. Los folds recientes pesan mas para priorizar la capacidad predictiva en el regimen post-COVID.

In [ ]:
# Cell 7 -- Grid Search Config Display
grid_rows = []
if "prophet" in loader.yamls:
    pg = loader.yamls["prophet"].get("param_grid_prophet", {})
    for pad_key, grid in pg.items():
        n = 1
        for vals in grid.values():
            n *= len(vals)
        grid_rows.append(
            {
                "Motor": "Prophet",
                "Padecimiento": pad_key.title(),
                "Parametros": ", ".join(f"{k}: {v}" for k, v in grid.items()),
                "Combinaciones": n,
            }
        )

if "ensemble" in loader.yamls:
    xg = loader.yamls["ensemble"].get("param_grid_xgboost", {})
    n = 1
    for vals in xg.values():
        n *= len(vals)
    grid_rows.append(
        {
            "Motor": "Ensemble (XGBoost)",
            "Padecimiento": "Todos",
            "Parametros": ", ".join(f"{k}: {v}" for k, v in xg.items()),
            "Combinaciones": n,
        }
    )

if "deepar" in loader.yamls:
    dg = loader.yamls["deepar"].get("param_grid_deepar", {})
    for pad_key, grid in dg.items():
        n = 1
        for vals in grid.values():
            n *= len(vals)
        grid_rows.append(
            {
                "Motor": "DeepAR",
                "Padecimiento": pad_key.title(),
                "Parametros": ", ".join(f"{k}: {v}" for k, v in grid.items()),
                "Combinaciones": n,
            }
        )

grid_rows.append(
    {
        "Motor": "Stacking",
        "Padecimiento": "Todos",
        "Parametros": "Prophet + ETS(52) + LightGBM(300,d=4); meta=ElasticNet(alpha=1.0, l1=0.5)",
        "Combinaciones": 1,
    }
)

df_grid = pd.DataFrame(grid_rows)
display(
    df_grid.style.set_properties(**{"text-align": "left", "font-size": "11px"})
    .set_table_styles(_imss_table_style())
    .hide(axis="index")
    .set_caption("Tabla 2. Busqueda de hiperparametros por motor y padecimiento")
)
total_combos = df_grid["Combinaciones"].sum()
print(
    f"\nTotal combinaciones evaluadas: {total_combos} x 111 series = {total_combos * 111:,} ejecuciones"
)

In [ ]:
# Cell 8 -- Violin plots de HPs seleccionados (2x2)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

pad_c = PAL.padecimiento


def _violin(ax: plt.Axes, data: list, labels: list, title: str, ylabel: str) -> None:
    if not data:
        ax.set_visible(False)
        return
    parts = ax.violinplot(data, showmeans=True, showmedians=True)
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(list(pad_c.values())[i % len(pad_c)])
        pc.set_alpha(0.6)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold", color=PAL.verde)
    ax.set_ylabel(ylabel)


# (0,0) Prophet cps
d, lbl = [], []
for pad in cfg.padecimientos:
    key = f"Prophet_{pad}"
    if key in loader.completos and "changepoint_prior_scale" in loader.completos[key].columns:
        d.append(loader.completos[key]["changepoint_prior_scale"].dropna().values)
        lbl.append(pad)
_violin(axes[0, 0], d, lbl, "Prophet: changepoint_prior_scale", "Valor")

# (0,1) Prophet sps
d, lbl = [], []
for pad in cfg.padecimientos:
    key = f"Prophet_{pad}"
    if key in loader.completos and "seasonality_prior_scale" in loader.completos[key].columns:
        d.append(loader.completos[key]["seasonality_prior_scale"].dropna().values)
        lbl.append(pad)
_violin(axes[0, 1], d, lbl, "Prophet: seasonality_prior_scale", "Valor")

# (1,0) Ensemble w_prophet
d, lbl = [], []
for pad in cfg.padecimientos:
    key = f"Ensemble_{pad}"
    if key in loader.completos and "w_prophet" in loader.completos[key].columns:
        d.append(loader.completos[key]["w_prophet"].dropna().values)
        lbl.append(pad)
_violin(axes[1, 0], d, lbl, "Ensemble: peso Prophet (w_prophet)", "Peso")
axes[1, 0].axhline(0.5, color=PAL.rojo, ls="--", alpha=0.5, label="50/50")
axes[1, 0].legend(fontsize=9)

# (1,1) Stacking pesos
expert_names = ["peso_prophet", "peso_ets", "peso_lgbm"]
expert_labels = ["Prophet", "ETS", "LightGBM"]
expert_colors = [PAL.prophet, PAL.stacking, PAL.ensemble]
d = []
for exp in expert_names:
    all_vals = []
    for pad in cfg.padecimientos:
        key = f"Stacking_{pad}"
        if key in loader.completos and exp in loader.completos[key].columns:
            all_vals.extend(loader.completos[key][exp].dropna().tolist())
    d.append(all_vals if all_vals else [0])
if any(len(v) > 1 for v in d):
    parts = axes[1, 1].violinplot(d, showmeans=True, showmedians=True)
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(expert_colors[i])
        pc.set_alpha(0.6)
    axes[1, 1].set_xticks(range(1, len(expert_labels) + 1))
    axes[1, 1].set_xticklabels(expert_labels, fontsize=10)
axes[1, 1].set_title(
    "Stacking: pesos de expertos (ElasticNet)", fontsize=12, fontweight="bold", color=PAL.verde
)
axes[1, 1].set_ylabel("Coeficiente")

fig.suptitle(
    "Distribucion de Hiperparametros Seleccionados",
    fontsize=15,
    fontweight="bold",
    color=PAL.verde,
    y=1.01,
)
fig.tight_layout()
guardar(fig, "08_violin_hiperparametros")
plt.show()
plt.close(fig)

In [ ]:
# Cell 9 -- Donut Prophet seasonality_mode
season_counts: dict[str, dict] = {}
for pad in cfg.padecimientos:
    key = f"Prophet_{pad}"
    if key in loader.completos and "seasonality_mode" in loader.completos[key].columns:
        season_counts[pad] = loader.completos[key]["seasonality_mode"].value_counts().to_dict()

if season_counts:
    fig, ax = plt.subplots(figsize=(8, 8))
    outer_labels, outer_sizes, outer_colors = [], [], []
    inner_labels, inner_sizes, inner_colors = [], [], []
    mode_alpha = {"additive": 0.5, "multiplicative": 0.85}
    for pad in cfg.padecimientos:
        if pad not in season_counts:
            continue
        total_pad = sum(season_counts[pad].values())
        inner_labels.append(pad)
        inner_sizes.append(total_pad)
        inner_colors.append(PAL.padecimiento[pad])
        for mode, count in season_counts[pad].items():
            outer_labels.append(f"{pad[:3]}: {mode[:5]}")
            outer_sizes.append(count)
            base_rgb = mcolors.to_rgb(PAL.padecimiento[pad])
            alpha = mode_alpha.get(mode, 0.7)
            outer_colors.append(tuple(c * alpha + 1.0 * (1 - alpha) for c in base_rgb))

    ax.pie(
        inner_sizes,
        labels=inner_labels,
        colors=inner_colors,
        radius=0.6,
        wedgeprops=dict(width=0.25, edgecolor="w"),
        textprops={"fontsize": 10, "fontweight": "bold"},
    )
    ax.pie(
        outer_sizes,
        labels=outer_labels,
        colors=outer_colors,
        radius=0.9,
        wedgeprops=dict(width=0.28, edgecolor="w"),
        textprops={"fontsize": 8},
        labeldistance=1.05,
    )
    ax.set_title(
        "Prophet: seasonality_mode por padecimiento\n(interno = padecimiento, externo = additive/multiplicative)",
        fontsize=12,
        fontweight="bold",
        color=PAL.verde,
    )
    fig.tight_layout()
    guardar(fig, "09_donut_seasonality")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 10 -- Scatter w_prophet vs SMAPE (metrica primaria)
fig, ax = plt.subplots(figsize=(10, 7))
all_w, all_s = [], []
for pad in cfg.padecimientos:
    key = f"Ensemble_{pad}"
    if key not in loader.completos:
        continue
    df = loader.completos[key]
    if "w_prophet" not in df.columns or "smape" not in df.columns:
        continue
    valid = df[["w_prophet", "smape"]].dropna()
    ax.scatter(
        valid["w_prophet"],
        valid["smape"],
        alpha=0.6,
        s=40,
        label=pad,
        color=PAL.padecimiento[pad],
        edgecolors="white",
        linewidth=0.5,
    )
    all_w.extend(valid["w_prophet"].tolist())
    all_s.extend(valid["smape"].tolist())
if all_w:
    z = np.polyfit(all_w, all_s, 1)
    x_line = np.linspace(0, 1, 50)
    r = np.corrcoef(all_w, all_s)[0, 1]
    ax.plot(
        x_line,
        np.polyval(z, x_line),
        "k--",
        alpha=0.4,
        linewidth=1.5,
        label=f"Tendencia (r={r:.2f})",
    )
ax.set_xlabel("Peso Prophet (w_prophet)", fontsize=12)
ax.set_ylabel("SMAPE (%)", fontsize=12)
ax.set_title(
    "Ensemble: relacion peso Prophet vs SMAPE", fontsize=13, fontweight="bold", color=PAL.verde
)
ax.legend(fontsize=9)
fig.tight_layout()
guardar(fig, "10_scatter_wprophet_smape")
plt.show()
plt.close(fig)

---

## **<font color="#004C46">ACT 3 -- Comparacion de Modelos</font>**

Evaluacion exhaustiva de los 4 motores. **Metrica primaria: SMAPE.** Desempate: MASE. Segundo desempate: RMSE. Se incluyen tiempos de entrenamiento como criterio operativo.

In [ ]:
# Cell 12 -- Tabla comparativa ordenada por SMAPE (metrica primaria)
rows_rank = []
for pad in cfg.padecimientos:
    det = loader.detalle[
        loader.detalle["Padecimiento"].str.contains(pad[:5], case=False, na=False)
    ]
    for modelo in cfg.modelos:
        victorias = 0
        if "Mejor MASE" in det.columns:
            victorias = int((det["Mejor MASE"] == modelo).sum())
        row = {"Padecimiento": cfg.pad_display[pad], "Modelo": modelo, "Victorias MASE": victorias}
        for metric in ["SMAPE", "MASE", "RMSE", "MAE"]:
            col = f"{metric} {modelo}"
            row[metric] = det[col].mean() if col in det.columns else np.nan
        t_col = f"Tiempo (s) {modelo}"
        row["Tiempo (min)"] = det[t_col].sum() / 60 if t_col in det.columns else np.nan
        rows_rank.append(row)

df_rank = pd.DataFrame(rows_rank)


def _highlight_smape_winner(df: pd.DataFrame) -> pd.DataFrame:
    """Resalta el ganador por SMAPE (metrica primaria) en dorado."""
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for pad in df["Padecimiento"].unique():
        mask = df["Padecimiento"] == pad
        idx_best = df.loc[mask, "SMAPE"].idxmin()
        if pd.notna(idx_best):
            styles.loc[idx_best] = "background-color: #FFF8E1; font-weight: bold"
    return styles


display(
    df_rank.style.format(
        {
            "SMAPE": "{:.2f}",
            "MASE": "{:.4f}",
            "RMSE": "{:.2f}",
            "MAE": "{:.2f}",
            "Tiempo (min)": "{:.1f}",
        }
    )
    .apply(_highlight_smape_winner, axis=None)
    .set_table_styles(_imss_table_style())
    .set_caption(
        "Tabla 3. Comparativa ordenada por SMAPE (metrica primaria) -- filas doradas = ganador"
    )
)

print("\n--- Ranking Global (promedio SMAPE) ---")
global_rank = (
    df_rank.groupby("Modelo")
    .agg(
        SMAPE_global=("SMAPE", "mean"),
        MASE_global=("MASE", "mean"),
        Victorias_total=("Victorias MASE", "sum"),
    )
    .sort_values("SMAPE_global")
)
display(
    global_rank.style.format({"SMAPE_global": "{:.2f}", "MASE_global": "{:.4f}"})
    .set_table_styles(_imss_table_style())
    .set_caption("Ranking Global por SMAPE")
)

In [ ]:
# Cell 13 -- Radar charts multi-metrica (1x3)
fig, axes = plt.subplots(1, 3, figsize=(20, 7), subplot_kw={"polar": True})
metrics_radar = ["SMAPE", "MASE", "RMSE", "MAE", "Tiempo"]

for idx, pad in enumerate(cfg.padecimientos):
    ax = axes[idx]
    det = loader.detalle[
        loader.detalle["Padecimiento"].str.contains(pad[:5], case=False, na=False)
    ]
    angles = np.linspace(0, 2 * np.pi, len(metrics_radar), endpoint=False).tolist()
    angles += angles[:1]

    for modelo in cfg.modelos:
        vals = []
        for m in metrics_radar:
            if m == "Tiempo":
                col = f"Tiempo (s) {modelo}"
                vals.append(det[col].sum() / 60 if col in det.columns else 0)
            else:
                col = f"{m} {modelo}"
                vals.append(det[col].mean() if col in det.columns else 0)
        max_val = max((v for v in vals if v > 0), default=1)
        vals_n = [v / max_val for v in vals] + [vals[0] / max_val]
        ax.plot(angles, vals_n, color=PAL.modelo[modelo], linewidth=1.5, label=modelo)
        ax.fill(angles, vals_n, color=PAL.modelo[modelo], alpha=0.08)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_radar, fontsize=9)
    ax.set_title(cfg.pad_display[pad], fontsize=12, fontweight="bold", color=PAL.verde, pad=20)
    if idx == 0:
        ax.legend(fontsize=8, loc="upper right", bbox_to_anchor=(0.1, 0.1))

fig.suptitle(
    "Radar Multi-Metrica por Padecimiento (menor area = mejor)",
    fontsize=14,
    fontweight="bold",
    color=PAL.verde,
    y=1.03,
)
fig.tight_layout()
guardar(fig, "13_radar_multimetrica")
plt.show()
plt.close(fig)

In [ ]:
# Cell 14 -- Violin plots SMAPE por modelo (metrica primaria)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for idx, pad in enumerate(cfg.padecimientos):
    ax = axes[idx]
    det = loader.detalle[
        loader.detalle["Padecimiento"].str.contains(pad[:5], case=False, na=False)
    ]
    violin_data, violin_labels = [], []
    for modelo in cfg.modelos:
        col = f"SMAPE {modelo}"
        if col in det.columns:
            vals = det[col].dropna().values
            if len(vals) > 0:
                violin_data.append(vals)
                violin_labels.append(modelo)
    if violin_data:
        parts = ax.violinplot(violin_data, showmeans=True, showmedians=True)
        for i, pc in enumerate(parts["bodies"]):
            pc.set_facecolor(PAL.modelo[violin_labels[i]])
            pc.set_alpha(0.6)
        ax.set_xticks(range(1, len(violin_labels) + 1))
        ax.set_xticklabels(violin_labels, fontsize=10)
    ax.set_title(cfg.pad_display[pad], fontsize=12, fontweight="bold", color=PAL.verde)
    if idx == 0:
        ax.set_ylabel("SMAPE (%)", fontsize=12)

fig.suptitle(
    "Distribucion de SMAPE por Modelo y Padecimiento (metrica primaria)",
    fontsize=14,
    fontweight="bold",
    color=PAL.verde,
    y=1.02,
)
fig.tight_layout()
guardar(fig, "14_violin_smape")
plt.show()
plt.close(fig)

In [ ]:
# Cell 15 -- Graficos overlay NACIONALES y REGIONALES (embebidos)
regiones = [
    "Region_Urbana_media",
    "Region_Sur-Sureste_vulnerable",
    "Region_Metropolitana_alta",
    "Region_Rural_-_dispersa",
]

for pad in cfg.padecimientos:
    print(f"\n{'=' * 70}")
    print(f"  {cfg.pad_display[pad]} -- PRONOSTICOS NACIONALES Y REGIONALES")
    print(f"{'=' * 70}\n")

    # Nacional general
    nac_path = _overlay_path(pad, "Nacional", "general")
    if nac_path and Path(nac_path).exists():
        print("  Nacional -- general:")
        display(Image(filename=nac_path, width=800))
    else:
        print("  Nacional general: no encontrado")

    # Regionales
    for region in regiones:
        reg_path = _overlay_path(pad, region, "general")
        if reg_path and Path(reg_path).exists():
            region_display = region.replace("Region_", "").replace("_", " ")
            print(f"\n  Region {region_display} -- general:")
            display(Image(filename=reg_path, width=800))

In [ ]:
# Cell 16 -- Residuales 4-panel (scatter + hist + QQ + ACF)
from scipy import stats as scipy_stats

for pad in cfg.padecimientos:
    serie = loader.load_serie_real(pad)
    y_col = "y_original" if "y_original" in serie.columns else "y"
    # Modelo con mejor SMAPE para este padecimiento
    best_model, best_smape = "Ensemble", float("inf")
    for modelo in cfg.modelos:
        col = f"SMAPE {modelo}"
        if col in loader.detalle.columns:
            det_pad = loader.detalle[
                loader.detalle["Padecimiento"].str.contains(pad[:5], case=False, na=False)
            ]
            s = det_pad[col].mean()
            if s < best_smape:
                best_smape, best_model = s, modelo

    fc_dict = loader.forecasts_nac.get(best_model, {})
    if pad not in fc_dict:
        print(f"  Sin datos {best_model} para {pad}")
        continue
    fc = fc_dict[pad]
    merged = (
        pd.DataFrame({"ds": serie["ds"], "y_real": serie[y_col]})
        .merge(fc[["ds", "yhat"]], on="ds", how="inner")
        .dropna()
    )
    merged["residual"] = merged["y_real"] - merged["yhat"]

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    # Scatter temporal
    axs[0, 0].scatter(
        merged["ds"], merged["residual"], s=10, alpha=0.5, color=PAL.modelo[best_model], zorder=3
    )
    axs[0, 0].axhline(0, color="black", linewidth=0.8, zorder=2)
    smooth = merged["residual"].rolling(12, min_periods=1, center=True).mean()
    axs[0, 0].plot(
        merged["ds"], smooth, color=PAL.rojo, linewidth=1.5, label="Media movil (12 sem)", zorder=4
    )
    axs[0, 0].set_title("Residuales vs Tiempo", fontweight="bold")
    axs[0, 0].legend(fontsize=9)

    # Histograma
    axs[0, 1].hist(
        merged["residual"],
        bins=30,
        color=PAL.modelo[best_model],
        alpha=0.7,
        edgecolor="white",
        density=True,
    )
    mu, sigma = merged["residual"].mean(), merged["residual"].std()
    x_norm = np.linspace(merged["residual"].min(), merged["residual"].max(), 100)
    axs[0, 1].plot(
        x_norm,
        (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_norm - mu) / sigma) ** 2),
        color=PAL.rojo,
        linewidth=2,
        label=f"N({mu:.1f}, {sigma:.1f})",
    )
    axs[0, 1].set_title("Distribucion de residuales", fontweight="bold")
    axs[0, 1].legend(fontsize=9)

    # QQ
    scipy_stats.probplot(merged["residual"].values, plot=axs[1, 0])
    axs[1, 0].set_title("Q-Q Normal", fontweight="bold")
    axs[1, 0].get_lines()[0].set_color(PAL.modelo[best_model])

    # ACF manual
    residuals = merged["residual"].values
    n = len(residuals)
    max_lag = min(40, n - 1)
    r_mean = residuals.mean()
    denom = np.sum((residuals - r_mean) ** 2)
    acf_vals = [
        np.sum((residuals[: n - lag] - r_mean) * (residuals[lag:] - r_mean)) / denom
        if denom > 0
        else 0
        for lag in range(max_lag + 1)
    ]
    axs[1, 1].bar(range(max_lag + 1), acf_vals, color=PAL.modelo[best_model], alpha=0.7)
    ci = 1.96 / np.sqrt(n)
    axs[1, 1].axhline(ci, ls="--", color=PAL.rojo, alpha=0.5)
    axs[1, 1].axhline(-ci, ls="--", color=PAL.rojo, alpha=0.5)
    axs[1, 1].set_title("Autocorrelacion (ACF)", fontweight="bold")
    axs[1, 1].set_xlabel("Lag (semanas)")

    fig.suptitle(
        f"Diagnostico de Residuales -- {cfg.pad_display[pad]} ({best_model})",
        fontsize=14,
        fontweight="bold",
        color=PAL.verde,
        y=1.01,
    )
    fig.tight_layout()
    guardar(fig, f"16_residuales_4panel_{pad.lower()}")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 17 -- Heatmap SMAPE por entidad x modelo (metrica primaria)
for pad in cfg.padecimientos:
    det = loader.detalle[
        loader.detalle["Padecimiento"].str.contains(pad[:5], case=False, na=False)
    ].copy()
    smape_cols = [f"SMAPE {m}" for m in cfg.modelos if f"SMAPE {m}" in det.columns]
    if not smape_cols:
        continue
    det["label"] = det.apply(
        lambda r: (str(r.get("Entidad", "Nacional")) if pd.notna(r.get("Entidad")) else "Nacional")
        + " | "
        + str(r.get("Sexo", "")),
        axis=1,
    )
    pivot = det.set_index("label")[smape_cols].dropna(how="all")
    pivot.columns = [c.replace("SMAPE ", "") for c in pivot.columns]
    if len(pivot) > 40:
        pivot = pivot.head(40)

    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot) * 0.28)))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1f",
        cmap="RdYlGn_r",
        linewidths=0.5,
        linecolor="white",
        ax=ax,
        vmin=0,
        vmax=min(200, pivot.max().max()),
        cbar_kws={"label": "SMAPE (%)", "shrink": 0.6},
    )
    ax.set_title(
        f"Mapa de Calor SMAPE -- {cfg.pad_display[pad]}",
        fontsize=13,
        fontweight="bold",
        color=PAL.verde,
        pad=15,
    )
    ax.set_xlabel("Modelo", fontsize=11)
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=8)
    fig.tight_layout()
    guardar(fig, f"17_heatmap_smape_{pad.lower()}")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 18 -- Ternario Stacking + Scatter Ensemble pesos
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(18, 8))

# Left: Ensemble w_prophet vs w_xgb
ew = []
for pad in cfg.padecimientos:
    key = f"Ensemble_{pad}"
    if key in loader.completos and "w_prophet" in loader.completos[key].columns:
        df = loader.completos[key]
        for _, row in df.iterrows():
            ew.append({"pad": pad, "w_prophet": row["w_prophet"], "w_xgb": row["w_xgb"]})
if ew:
    df_ew = pd.DataFrame(ew)
    for pad in cfg.padecimientos:
        sub = df_ew[df_ew["pad"] == pad]
        ax_left.scatter(
            sub["w_prophet"], sub["w_xgb"], alpha=0.5, s=30, label=pad, color=PAL.padecimiento[pad]
        )
    ax_left.plot([0, 1], [1, 0], "k--", alpha=0.3)
    ax_left.set_xlabel("Peso Prophet", fontsize=11)
    ax_left.set_ylabel("Peso XGBoost", fontsize=11)
    ax_left.set_title(
        "Ensemble: pesos Prophet vs XGBoost", fontsize=12, fontweight="bold", color=PAL.verde
    )
    ax_left.legend(fontsize=9)
    ax_left.set_xlim(-0.05, 1.05)
    ax_left.set_ylim(-0.05, 1.05)

# Right: Ternario Stacking (manual)
sw = []
for pad in cfg.padecimientos:
    key = f"Stacking_{pad}"
    if key in loader.completos:
        df = loader.completos[key]
        w_cols = ["peso_prophet", "peso_ets", "peso_lgbm"]
        if all(c in df.columns for c in w_cols):
            for _, row in df.iterrows():
                total = sum(abs(row[c]) for c in w_cols)
                if total > 0:
                    sw.append(
                        {
                            "pad": pad,
                            "a": abs(row["peso_prophet"]) / total,
                            "b": abs(row["peso_ets"]) / total,
                            "c": abs(row["peso_lgbm"]) / total,
                        }
                    )
if sw:
    df_sw = pd.DataFrame(sw)
    df_sw["x"] = 0.5 * (2 * df_sw["b"] + df_sw["c"])
    df_sw["y_t"] = (np.sqrt(3) / 2) * df_sw["c"]
    ax_right.plot([0, 1, 0.5, 0], [0, 0, np.sqrt(3) / 2, 0], "k-", linewidth=1.5)
    for pad in cfg.padecimientos:
        sub = df_sw[df_sw["pad"] == pad]
        ax_right.scatter(
            sub["x"], sub["y_t"], alpha=0.6, s=40, label=pad, color=PAL.padecimiento[pad]
        )
    ax_right.text(-0.05, -0.05, "Prophet", fontsize=10, fontweight="bold", ha="center")
    ax_right.text(1.05, -0.05, "ETS", fontsize=10, fontweight="bold", ha="center")
    ax_right.text(
        0.5, np.sqrt(3) / 2 + 0.05, "LightGBM", fontsize=10, fontweight="bold", ha="center"
    )
    ax_right.set_title(
        "Stacking: diagrama ternario de pesos", fontsize=12, fontweight="bold", color=PAL.verde
    )
    ax_right.legend(fontsize=9)
    ax_right.set_xlim(-0.15, 1.15)
    ax_right.set_ylim(-0.15, 1.05)
    ax_right.set_aspect("equal")
    ax_right.axis("off")

fig.tight_layout()
guardar(fig, "18_ternario_pesos")
plt.show()
plt.close(fig)

In [ ]:
# Cell 19 -- Feature Importance XGBoost (barras + error bars)
feature_data: dict[str, np.ndarray] = {}
feature_names: list[str] = []
for pad in cfg.padecimientos:
    pkl_dir = cfg.models_dir / "ensemble" / pad
    if not pkl_dir.exists():
        continue
    for pkl_file in sorted(pkl_dir.glob("*.pkl")):
        if "completo" in pkl_file.name:
            continue
        try:
            with pkl_file.open("rb") as f:
                data = pickle.load(f)
            pe = data.get("parallel_engine")
            if pe is None:
                continue
            xgb_direct = getattr(pe, "xgb_direct", None)
            if xgb_direct is None:
                continue
            inner = getattr(xgb_direct, "_model", None)
            if inner is None or not hasattr(inner, "feature_importances_"):
                continue
            imp = inner.feature_importances_
            if not feature_names:
                feature_names = list(
                    getattr(inner, "feature_names_in_", [f"f{i}" for i in range(len(imp))])
                )
            feature_data[pkl_file.stem] = imp
        except Exception:
            continue

if feature_data and feature_names:
    all_imp = np.stack(list(feature_data.values()))
    mean_imp, std_imp = all_imp.mean(axis=0), all_imp.std(axis=0)
    order = np.argsort(mean_imp)[::-1]
    sorted_names = [feature_names[i] for i in order]
    sorted_mean, sorted_std = mean_imp[order], std_imp[order]
    cat_colors = [
        PAL.prophet if "lag" in n else PAL.ensemble if "roll" in n else PAL.stacking
        for n in sorted_names
    ]

    fig, ax = plt.subplots(figsize=(12, 7))
    y_pos = np.arange(len(sorted_names))
    ax.barh(y_pos, sorted_mean, xerr=sorted_std, color=cat_colors, alpha=0.85, edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_names, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel("Importancia relativa (gain)", fontsize=11)
    ax.set_title(
        f"Importancia de Features -- XGBoost ({len(feature_data)} modelos)",
        fontsize=13,
        fontweight="bold",
        color=PAL.verde,
    )
    ax.legend(
        handles=[
            Patch(facecolor=PAL.prophet, label="Lags"),
            Patch(facecolor=PAL.ensemble, label="Rolling"),
            Patch(facecolor=PAL.stacking, label="Ciclicos"),
        ],
        fontsize=9,
        loc="lower right",
    )
    fig.tight_layout()
    guardar(fig, "19_feature_importance")
    plt.show()
    plt.close(fig)
    print(f"Features de {len(feature_data)} modelos Ensemble con XGBoost.")
else:
    print("Sin modelos Ensemble con XGBoost entrenado.")

In [ ]:
# Cell 20 -- Stacking meta-learner coeficientes
coef_data: dict[str, dict] = {}
for pad in cfg.padecimientos:
    key = f"Stacking_{pad}"
    if key in loader.completos:
        df = loader.completos[key]
        w_cols = ["peso_prophet", "peso_ets", "peso_lgbm"]
        if all(c in df.columns for c in w_cols):
            coef_data[pad] = {
                "Prophet": df["peso_prophet"].mean(),
                "ETS": df["peso_ets"].mean(),
                "LightGBM": df["peso_lgbm"].mean(),
            }

if coef_data:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(3)
    width = 0.25
    for i, pad in enumerate(cfg.padecimientos):
        if pad in coef_data:
            ax.bar(
                x + i * width,
                list(coef_data[pad].values()),
                width,
                label=pad,
                alpha=0.8,
                color=PAL.padecimiento[pad],
            )
    ax.set_xticks(x + width)
    ax.set_xticklabels(["Prophet", "ETS", "LightGBM"], fontsize=11)
    ax.set_ylabel("Coeficiente promedio (ElasticNet)", fontsize=11)
    ax.set_title(
        "Stacking: coeficientes del meta-learner por padecimiento",
        fontsize=13,
        fontweight="bold",
        color=PAL.verde,
    )
    ax.legend(fontsize=10)
    ax.axhline(0, color="black", linewidth=0.5)
    fig.tight_layout()
    guardar(fig, "20_stacking_coeficientes")
    plt.show()
    plt.close(fig)

---

## **<font color="#004C46">ACT 4 -- Modelo de Produccion (Tabla Completa de 333 Modelos)</font>**

La tabla de produccion selecciona automaticamente el mejor modelo para cada una de las 333 combinaciones (padecimiento x entidad x sexo) usando el criterio:

1. **SMAPE** como metrica primaria
2. **MASE** como desempate (umbral 5%)
3. **RMSE** como segundo desempate

Incluye diagnosticos de overfitting (ratio smape_test/smape_train) y leakage (smape_train sospechosamente bajo).

In [ ]:
# Cell 22 -- TABLA COMPLETA DE 333 MODELOS DE PRODUCCION
if not loader.produccion.empty:
    prod = loader.produccion.copy()
    # Construir tabla de exhibicion
    display_cols = [
        "numero",
        "padecimiento",
        "entidad",
        "sexo",
        "modelo_produccion",
        "tipo_modelo",
        "region_asignada",
        "smape_prod",
        "mase_prod",
        "rmse_prod",
        "mae_prod",
        "overfitting",
        "leakage",
        "casos_52_semanas_futuro",
        "justificacion",
    ]
    available = [c for c in display_cols if c in prod.columns]
    df_display = prod[available].copy()

    # Renombrar columnas para legibilidad
    rename_map = {
        "numero": "#",
        "padecimiento": "Padecimiento",
        "entidad": "Entidad",
        "sexo": "Sexo",
        "modelo_produccion": "Modelo Prod.",
        "tipo_modelo": "Tipo",
        "region_asignada": "Region",
        "smape_prod": "SMAPE%",
        "mase_prod": "MASE",
        "rmse_prod": "RMSE",
        "mae_prod": "MAE",
        "overfitting": "Overfitting",
        "leakage": "Leakage",
        "casos_52_semanas_futuro": "Casos 52sem",
        "justificacion": "Justificacion",
    }
    df_display = df_display.rename(
        columns={k: v for k, v in rename_map.items() if k in df_display.columns}
    )

    # Color por modelo productivo
    def _color_modelo(val: str) -> str:
        color_map = {
            "DeepAR": f"background-color: {PAL.deepar}22; color: {PAL.deepar}",
            "Prophet": f"background-color: {PAL.prophet}22; color: {PAL.prophet}",
            "Ensemble": f"background-color: {PAL.ensemble}22; color: {PAL.ensemble}",
            "Stacking": f"background-color: {PAL.stacking}22; color: {PAL.stacking}",
        }
        return color_map.get(str(val), "")

    fmt = {}
    if "SMAPE%" in df_display.columns:
        fmt["SMAPE%"] = "{:.1f}"
    if "MASE" in df_display.columns:
        fmt["MASE"] = "{:.4f}"
    if "RMSE" in df_display.columns:
        fmt["RMSE"] = "{:.2f}"
    if "MAE" in df_display.columns:
        fmt["MAE"] = "{:.2f}"
    if "Casos 52sem" in df_display.columns:
        fmt["Casos 52sem"] = "{:.0f}"

    styler = (
        df_display.style.format(fmt, na_rep="--")
        .set_table_styles(_imss_table_style())
        .set_properties(**{"font-size": "10px", "text-align": "left"})
        .set_caption(
            "Tabla 4. Los 333 modelos de produccion -- casuistica completa, modelo seleccionado y justificacion"
        )
    )
    if "Modelo Prod." in df_display.columns:
        styler = styler.map(_color_modelo, subset=["Modelo Prod."])
    display(styler)

    # Resumen
    pc = prod["modelo_produccion"].value_counts()
    print("\n--- Distribucion de modelos productivos ---")
    for m, c in pc.items():
        print(f"  {m}: {c} ({c / len(prod) * 100:.1f}%)")
    print(f"  Total: {len(prod)} series")
else:
    print("Tabla de produccion no disponible.")

In [ ]:
# Cell 23 -- Donut chart distribucion produccion
if not loader.produccion.empty:
    prod_counts = loader.produccion["modelo_produccion"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 8))
    colors_prod = [PAL.modelo.get(m, "#888") for m in prod_counts.index]
    wedges, texts, autotexts = ax.pie(
        prod_counts.values,
        labels=[f"{m}\n({c}, {c / 333 * 100:.1f}%)" for m, c in prod_counts.items()],
        colors=colors_prod,
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops=dict(width=0.4, edgecolor="white"),
        textprops={"fontsize": 11},
        pctdistance=0.78,
    )
    for at in autotexts:
        at.set_fontsize(10)
        at.set_fontweight("bold")
        at.set_color("white")
    ax.set_title(
        "Distribucion del Modelo de Produccion\n(333 series, seleccion automatica por SMAPE)",
        fontsize=13,
        fontweight="bold",
        color=PAL.verde,
    )
    fig.tight_layout()
    guardar(fig, "23_donut_produccion")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 24 -- Stacked bar produccion por padecimiento
if not loader.produccion.empty:
    cross = pd.crosstab(loader.produccion["padecimiento"], loader.produccion["modelo_produccion"])
    col_order = [m for m in cfg.modelos if m in cross.columns]
    cross = cross[col_order]

    fig, ax = plt.subplots(figsize=(12, 6))
    cross.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[PAL.modelo.get(m, "#888") for m in col_order],
        edgecolor="white",
        linewidth=0.5,
    )
    ax.set_ylabel("Numero de series", fontsize=11)
    ax.set_xlabel("")
    ax.set_title(
        "Modelo de produccion por padecimiento (seleccion automatica SMAPE)",
        fontsize=13,
        fontweight="bold",
        color=PAL.verde,
    )
    ax.legend(fontsize=10, title="Motor")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
    for container in ax.containers:
        labels = [f"{int(v.get_height())}" if v.get_height() > 0 else "" for v in container]
        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            fontsize=9,
            color="white",
            fontweight="bold",
        )
    fig.tight_layout()
    guardar(fig, "24_stacked_bar_produccion")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 25 -- Scatter overfitting diagnostics
if not loader.produccion.empty:
    prod = loader.produccion.copy()
    prod["of_ratio"] = prod["overfitting"].str.extract(r"([\d.]+)x", expand=False).astype(float)

    fig, ax = plt.subplots(figsize=(10, 8))
    for modelo in cfg.modelos:
        mask = prod["modelo_produccion"] == modelo
        sub = prod[mask]
        ax.scatter(
            sub["smape_prod"],
            sub["of_ratio"],
            alpha=0.5,
            s=30,
            color=PAL.modelo.get(modelo, "#888"),
            label=f"{modelo} ({mask.sum()})",
            edgecolors="white",
            linewidth=0.3,
        )
    ax.axhline(1.3, color=PAL.ensemble, ls="--", alpha=0.6, label="Moderado (1.3x)")
    ax.axhline(2.0, color=PAL.rojo, ls="--", alpha=0.6, label="Alto (2.0x)")
    ax.set_xlabel("SMAPE produccion (%)", fontsize=12)
    ax.set_ylabel("Ratio overfitting (test/train)", fontsize=12)
    ax.set_title(
        "Diagnostico de Overfitting -- 333 Modelos de Produccion",
        fontsize=13,
        fontweight="bold",
        color=PAL.verde,
    )
    ax.legend(fontsize=9, loc="upper right")
    fig.tight_layout()
    guardar(fig, "25_scatter_overfitting")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 26 -- Top 5 y Bottom 5 con PNGs embebidos de overlay
if not loader.produccion.empty:
    prod = loader.produccion.copy()

    # TOP 5 por SMAPE (menor = mejor)
    top5 = prod.nsmallest(5, "smape_prod")
    print("=" * 70)
    print("  TOP 5 MEJORES MODELOS DE PRODUCCION (menor SMAPE)")
    print("=" * 70)
    for _, row in top5.iterrows():
        ent = row["entidad"] if pd.notna(row["entidad"]) else "Nacional"
        pad_key = row.get("pad_key", row["padecimiento"])
        print(
            f"\n  {row['padecimiento']} | {ent} | {row['sexo']}"
            f" | {row['modelo_produccion']}"
            f" | SMAPE={row['smape_prod']:.1f}%"
            f" | MASE={row['mase_prod']:.4f}"
            if pd.notna(row["mase_prod"])
            else ""
        )
        png = _overlay_path(pad_key, ent, row["sexo"] if pd.notna(row["sexo"]) else "")
        if png and Path(png).exists():
            display(Image(filename=png, width=700))
        else:
            print("    (Grafico overlay no encontrado)")

    # BOTTOM 5 por SMAPE (mayor = peor, excluir regionales sin incidencia)
    prod_nonzero = prod[prod["smape_prod"] > 0]
    bot5 = prod_nonzero.nlargest(5, "smape_prod")
    print(f"\n{'=' * 70}")
    print("  BOTTOM 5 MODELOS DE PRODUCCION (mayor SMAPE)")
    print("=" * 70)
    for _, row in bot5.iterrows():
        ent = row["entidad"] if pd.notna(row["entidad"]) else "Nacional"
        pad_key = row.get("pad_key", row["padecimiento"])
        print(
            f"\n  {row['padecimiento']} | {ent} | {row['sexo']}"
            f" | {row['modelo_produccion']}"
            f" | SMAPE={row['smape_prod']:.1f}%"
            f" | MASE={row['mase_prod']:.4f}"
            if pd.notna(row["mase_prod"])
            else ""
        )
        png = _overlay_path(pad_key, ent, row["sexo"] if pd.notna(row["sexo"]) else "")
        if png and Path(png).exists():
            display(Image(filename=png, width=700))
        else:
            print("    (Grafico overlay no encontrado)")

In [ ]:
# Cell 27 -- Matriz de seleccion ponderada (SMAPE primaria)
global_m: dict[str, dict] = {}
for modelo in cfg.modelos:
    smape_vals, mase_vals, rmse_vals, vic = [], [], [], 0
    for pad in cfg.padecimientos:
        det = loader.detalle[
            loader.detalle["Padecimiento"].str.contains(pad[:5], case=False, na=False)
        ]
        for metric, lst in [("SMAPE", smape_vals), ("MASE", mase_vals), ("RMSE", rmse_vals)]:
            col = f"{metric} {modelo}"
            if col in det.columns:
                lst.extend(det[col].dropna().tolist())
        if "Mejor MASE" in det.columns:
            vic += int((det["Mejor MASE"] == modelo).sum())
    global_m[modelo] = {
        "SMAPE": np.mean(smape_vals) if smape_vals else 0,
        "MASE": np.mean(mase_vals) if mase_vals else 0,
        "RMSE": np.mean(rmse_vals) if rmse_vals else 0,
        "Victorias": vic,
    }


def _norm_min(vals: list[float]) -> list[float]:
    mn, mx = min(vals), max(vals)
    return [10 * (1 - (v - mn) / (mx - mn)) if mx > mn else 5 for v in vals]


def _norm_max(vals: list[float]) -> list[float]:
    mn, mx = min(vals), max(vals)
    return [10 * (v - mn) / (mx - mn) if mx > mn else 5 for v in vals]


models_list = list(cfg.modelos)
# SMAPE tiene peso principal (40%), MASE como desempate (20%)
scores = np.array(
    [
        _norm_min([global_m[m]["SMAPE"] for m in models_list]),  # SMAPE primaria
        _norm_min([global_m[m]["MASE"] for m in models_list]),  # MASE desempate
        _norm_min([global_m[m]["RMSE"] for m in models_list]),  # RMSE
        _norm_max([global_m[m]["Victorias"] for m in models_list]),
        [7.0, 5.0, 8.0, 6.0],
    ]
).T
weights = np.array([0.40, 0.20, 0.10, 0.15, 0.15])
totals = (scores * weights).sum(axis=1)

criteria = ["SMAPE (40%)", "MASE (20%)", "RMSE (10%)", "Victorias (15%)", "Interpretab. (15%)"]
df_sel = pd.DataFrame(scores, index=models_list, columns=criteria)
df_sel["TOTAL"] = totals
display(
    df_sel.style.format("{:.2f}")
    .background_gradient(cmap="YlGn", subset=["TOTAL"])
    .set_table_styles(_imss_table_style())
    .set_caption("Tabla 5. Matriz de seleccion ponderada -- SMAPE primaria, MASE desempate")
)

winner = models_list[int(np.argmax(totals))]
print(f"\nModelo seleccionado (matriz): {winner} (puntaje: {max(totals):.2f}/10)")
print(f"  SMAPE global: {global_m[winner]['SMAPE']:.2f}%")
print(f"  MASE global:  {global_m[winner]['MASE']:.4f}")
print(f"  Victorias:    {global_m[winner]['Victorias']}/441")

if not loader.produccion.empty:
    pc = loader.produccion["modelo_produccion"].value_counts()
    print("\n--- Seleccion real de produccion (333 series, por SMAPE) ---")
    for m, c in pc.items():
        print(f"  {m}: {c} ({c / 333 * 100:.1f}%)")

---

## **<font color="#7B2132">Veredicto: Modelo de Produccion</font>**

<div style="border: 3px solid #DAA520; border-radius: 12px; padding: 20px; background-color: #FFF8E1;">

Tras evaluar 1,332 modelos (333 por motor x 4 motores) sobre 3 padecimientos, 32 entidades federativas, 4 regiones INEGI y 3 modos de desagregacion por sexo, la **seleccion automatica por SMAPE** (con MASE como desempate) asigna el modelo de produccion individualmente para cada serie:

- **DeepAR** domina con 244 series (73.3%), gracias al *transfer learning* multi-serie.
- **Prophet** captura 55 series (16.5%), donde la estacionalidad Fourier es optima.
- **Ensemble** gana 32 series (9.6%), donde Prophet+XGBoost aporta robustez.
- **Stacking** aporta 2 series (0.6%), donde la diversidad de expertos es superior.

El 99.4% de los modelos presenta **overfitting OK** (<1.3x) y solo 3 series muestran leakage sospechoso (smape_train=0%), todas de incidencia cero.

Para el dashboard de alerta temprana del IMSS, se despliega el **modelo productivo por serie**, seleccionado automaticamente por la tabla de 333 modelos.

</div>

In [ ]:
# Cell 29 -- Resumen numerico de produccion
if not loader.produccion.empty:
    prod = loader.produccion
    print("=" * 60)
    print("  RESUMEN NUMERICO DE PRODUCCION")
    print("=" * 60)
    print(f"  Total series:            {len(prod)}")
    print(f"  SMAPE medio:             {prod['smape_prod'].mean():.2f}%")
    print(f"  MASE medio:              {prod['mase_prod'].mean():.4f}")
    print(f"  RMSE medio:              {prod['rmse_prod'].mean():.2f}")
    n_below1 = (prod["mase_prod"] < 1.0).sum()
    print(f"  Series MASE < 1.0:       {n_below1} ({n_below1 / len(prod) * 100:.1f}%)")
    n_ok = prod["overfitting"].str.contains("OK", na=False).sum()
    print(f"  Overfitting OK:          {n_ok} ({n_ok / len(prod) * 100:.1f}%)")
    if "casos_52_semanas_futuro" in prod.columns:
        print(
            f"  Casos pronosticados:     {int(prod['casos_52_semanas_futuro'].sum()):,} (52 sem)"
        )

---

## **<font color="#004C46">ACT 5 -- Dashboard Ejecutivo</font>**

Panel de indicadores clave de rendimiento (KPIs) del sistema EpiForecast-MX.

In [ ]:
# Cell 31 -- KPI Cards dashboard 2x3
kpi_data = []
if not loader.produccion.empty:
    prod = loader.produccion
    n_below1 = (prod["mase_prod"] < 1.0).sum()
    pct_below = n_below1 / len(prod) * 100
    top_motor = prod["modelo_produccion"].value_counts().index[0]
    top_pct = prod["modelo_produccion"].value_counts().iloc[0] / len(prod) * 100
    kpi_data = [
        ("1,332", "Modelos\nentrenados", PAL.verde),
        (f"{prod['smape_prod'].mean():.1f}%", "SMAPE\npromedio", PAL.ensemble),
        (f"{top_pct:.1f}%", f"{top_motor}\nen produccion", PAL.deepar),
        (f"{pct_below:.0f}%", "Series\n< naive", PAL.prophet),
        (f"{prod['mase_prod'].mean():.2f}", "MASE\npromedio", PAL.stacking),
        ("52", "Semanas\nhorizonte", PAL.rojo),
    ]

if kpi_data:
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    for i, (value, label, color) in enumerate(kpi_data):
        ax = axes.flat[i]
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")
        rect = FancyBboxPatch(
            (0.05, 0.05),
            0.9,
            0.9,
            boxstyle="round,pad=0.05",
            facecolor=color,
            alpha=0.12,
            edgecolor=color,
            linewidth=2,
        )
        ax.add_patch(rect)
        ax.text(
            0.5, 0.60, value, fontsize=36, fontweight="bold", color=color, ha="center", va="center"
        )
        ax.text(0.5, 0.25, label, fontsize=13, color=PAL.gris_oscuro, ha="center", va="center")
    fig.suptitle(
        "Dashboard Ejecutivo -- EpiForecast-MX",
        fontsize=16,
        fontweight="bold",
        color=PAL.verde,
        y=1.01,
    )
    fig.tight_layout()
    guardar(fig, "31_kpi_dashboard")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 32 -- Matriz entidad x padecimiento: modelo ganador (modo general)
if not loader.produccion.empty:
    prod = loader.produccion.copy()
    prod_gen = prod[
        prod["sexo"].astype(str).str.contains("total|general|General", case=False, na=False)
    ].copy()
    if prod_gen.empty:
        prod_gen = prod.copy()

    model_map = {"Prophet": 0, "DeepAR": 1, "Ensemble": 2, "Stacking": 3}
    prod_gen["model_num"] = prod_gen["modelo_produccion"].map(model_map)
    pivot = prod_gen.pivot_table(
        index="entidad", columns="padecimiento", values="model_num", aggfunc="first"
    ).dropna(how="all")
    model_names = list(model_map.keys())
    cmap = mcolors.ListedColormap([PAL.modelo.get(m, "#888") for m in model_names])
    bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=(8, max(6, len(pivot) * 0.3)))
    ax.imshow(pivot.values, cmap=cmap, norm=norm, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=10)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.legend(
        handles=[Patch(facecolor=PAL.modelo.get(m, "#888"), label=m) for m in model_names],
        fontsize=9,
        loc="upper right",
        bbox_to_anchor=(1.35, 1),
    )
    ax.set_title(
        "Modelo ganador por entidad y padecimiento (modo general)",
        fontsize=13,
        fontweight="bold",
        color=PAL.verde,
    )
    fig.tight_layout()
    guardar(fig, "32_matriz_ganador")
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 33 -- SMAPE vs Tiempo: frontera de eficiencia
fig, ax = plt.subplots(figsize=(10, 7))
for modelo in cfg.modelos:
    s_col = f"SMAPE {modelo}"
    t_col = f"Tiempo (s) {modelo}"
    if s_col not in loader.detalle.columns or t_col not in loader.detalle.columns:
        continue
    smape_vals = loader.detalle[s_col].dropna()
    time_vals = loader.detalle[t_col].dropna()
    ax.scatter(
        time_vals.mean() / 60,
        smape_vals.mean(),
        s=200,
        color=PAL.modelo[modelo],
        edgecolors="black",
        linewidth=1.5,
        label=modelo,
        zorder=5,
    )
    ax.errorbar(
        time_vals.mean() / 60,
        smape_vals.mean(),
        xerr=time_vals.std() / 60,
        yerr=smape_vals.std(),
        color=PAL.modelo[modelo],
        alpha=0.3,
        capsize=5,
    )
ax.set_xlabel("Tiempo medio de entrenamiento (min)", fontsize=12)
ax.set_ylabel("SMAPE medio (%)", fontsize=12)
ax.set_title(
    "Frontera de Eficiencia: Precision (SMAPE) vs Costo Computacional",
    fontsize=13,
    fontweight="bold",
    color=PAL.verde,
)
ax.legend(fontsize=10)
fig.tight_layout()
guardar(fig, "33_frontera_eficiencia")
plt.show()
plt.close(fig)

In [ ]:
# Cell 34 -- Mejora ensemble vs individual (paired comparison)
mejora_rows = []
for pad in cfg.padecimientos:
    det = loader.detalle[
        loader.detalle["Padecimiento"].str.contains(pad[:5], case=False, na=False)
    ]
    for _, row in det.iterrows():
        p_smape = row.get("SMAPE Prophet", np.nan)
        e_smape = row.get("SMAPE Ensemble", np.nan)
        if pd.notna(p_smape) and pd.notna(e_smape) and p_smape > 0:
            mejora_rows.append(
                {
                    "pad": pad,
                    "Prophet": p_smape,
                    "Ensemble": e_smape,
                    "mejora_pct": (p_smape - e_smape) / p_smape * 100,
                }
            )

if mejora_rows:
    df_mej = pd.DataFrame(mejora_rows)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    for pad in cfg.padecimientos:
        sub = df_mej[df_mej["pad"] == pad]
        ax1.scatter(
            sub["Prophet"],
            sub["Ensemble"],
            alpha=0.5,
            s=25,
            label=pad,
            color=PAL.padecimiento[pad],
        )
    lim = max(df_mej["Prophet"].max(), df_mej["Ensemble"].max()) * 1.05
    ax1.plot([0, lim], [0, lim], "k--", alpha=0.3, label="Sin mejora")
    ax1.set_xlabel("SMAPE Prophet (%)", fontsize=11)
    ax1.set_ylabel("SMAPE Ensemble (%)", fontsize=11)
    ax1.set_title(
        "Prophet vs Ensemble (bajo diagonal = Ensemble mejor)",
        fontsize=12,
        fontweight="bold",
        color=PAL.verde,
    )
    ax1.legend(fontsize=9)

    ax2.hist(
        df_mej["mejora_pct"].dropna(), bins=30, color=PAL.ensemble, alpha=0.7, edgecolor="white"
    )
    ax2.axvline(0, color="black", ls="--", alpha=0.5)
    ax2.axvline(
        df_mej["mejora_pct"].mean(),
        color=PAL.rojo,
        ls="-",
        linewidth=2,
        label=f"Media: {df_mej['mejora_pct'].mean():.1f}%",
    )
    ax2.set_xlabel("Mejora Ensemble vs Prophet (% SMAPE)", fontsize=11)
    ax2.set_ylabel("Frecuencia", fontsize=11)
    ax2.set_title(
        "Distribucion de mejora del Ensemble", fontsize=12, fontweight="bold", color=PAL.verde
    )
    ax2.legend(fontsize=10)

    fig.tight_layout()
    guardar(fig, "34_mejora_ensemble")
    plt.show()
    plt.close(fig)

---

## **<font color="#004C46">Conclusiones</font>**

1. **Cuatro estrategias de ensemble implementadas y evaluadas.** Prophet (baseline), DeepAR (homogeneo multi-serie), Ensemble Paralelo (heterogeneo ponderado OOF) y Stacking (heterogeneo con ElasticNet) cubren el espectro completo, cumpliendo el Objetivo 3.5.

2. **SMAPE como metrica primaria valida la seleccion automatica.** Cada una de las 333 series tiene asignado el modelo con menor SMAPE, con MASE como desempate y RMSE como segundo criterio (Tabla 4, ACT 4).

3. **DeepAR domina en produccion con 73.3% de las series.** El entrenamiento multi-serie permite transferir patrones entre las 32 entidades, beneficiando especialmente series de baja incidencia como Alzheimer en estados pequenos.

4. **La optimizacion de hiperparametros es especifica por padecimiento.** Prophet usa grids de 6-24 combinaciones; XGBoost evalua 36 configuraciones con early stopping; DeepAR optimiza epochs y capas (ACT 2).

5. **Los pesos del Ensemble revelan complementariedad Prophet-XGBoost.** La distribucion de w_prophet muestra variabilidad segun la serie, confirmando que ambos componentes aportan informacion no redundante.

6. **lag_52 es el predictor mas importante del XGBoost.** La estacionalidad anual de 52 semanas es el patron dominante en las 3 enfermedades cronicas.

7. **99.4% de modelos sin overfitting.** Solo 1 serie muestra overfitting moderado y 1 alto. El diagnostico de leakage identifica 3 series sospechosas, todas de incidencia cero.

8. **La tabla completa de 333 modelos ofrece trazabilidad total.** Cada serie incluye justificacion, metricas de produccion, diagnosticos y el modelo seleccionado, cumpliendo el Objetivo 3.6.

---

## **<font color="#004C46">Referencias</font>**

| Referencia | Tema |
|:-----------|:-----|
| Makridakis, S. (1993). Accuracy measures: theoretical and practical concerns. *IJF*, 9(4). | SMAPE como metrica primaria. |
| Hyndman, R. & Koehler, A. (2006). Another look at measures of forecast accuracy. *IJF*, 22(4). | MASE como metrica de desempate. |
| Singh, A. (2023). *Ensemble Learning: A Practitioner's Guide*. | Taxonomia homogeneo/heterogeneo. |
| Salinas, D. et al. (2020). DeepAR: Probabilistic forecasting with autoregressive recurrent networks. *IJF*, 36(3). | Arquitectura DeepAR. |
| Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD 2016*. | Gradient boosting. |
| Taylor, S. & Letham, B. (2018). Forecasting at Scale. *The American Statistician*, 72(1). | Prophet. |
| Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *NIPS 2017*. | LightGBM en Stacking. |
| Zou, H. & Hastie, T. (2005). Regularization and variable selection via the elastic net. *JRSS-B*, 67(2). | ElasticNet meta-learner. |
| VanderPlas, J. (2022). *Python Data Science Handbook*, Cap. 5. | Validacion cruzada. |

In [ ]:
# Cell 37 -- Export figuras + ZIP
figuras = sorted(cfg.fig_dir.glob("*.png"))
print(f"Total figuras generadas: {len(figuras)}")
for f in figuras:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:6.0f} KB")

zip_path = cfg.fig_dir.parent / "figuras_avance5_latex"
if zip_path.with_suffix(".zip").exists():
    zip_path.with_suffix(".zip").unlink()
shutil.make_archive(str(zip_path), "zip", cfg.fig_dir)
print(f"\nZIP creado: {zip_path.with_suffix('.zip')}")

---

## **<font color="#004C46">Apendice: Glosario de Metricas</font>**

| Metrica | Formula | Interpretacion |
|:--------|:--------|:---------------|
| **SMAPE** | 200 * \|y-yhat\| / (\|y\|+\|yhat\|) | 0% = perfecto, simetrico. **Metrica primaria.** |
| **MASE** | MAE / MAE_naive_seasonal(52) | < 1.0 = mejor que naive. **Desempate.** |
| **RMSE** | sqrt(mean((y-yhat)^2)) | Penaliza errores grandes. Segundo desempate. |
| **MAE** | mean(\|y-yhat\|) | Error absoluto medio. |
| **Overfitting** | SMAPE_test / SMAPE_train | >1.3x moderado, >2.0x alto. |
| **Leakage** | SMAPE_train < 0.5% | Sospechoso de fuga de datos. |
| **Precision historica** | sum(yhat) / sum(y_real) * 100 | 100% = perfecto. |

---

<center>
<small style="color: #999;">
EpiForecast-MX v5.0 | Avance 5 -- Modelo Final | Marzo 2026<br>
Notebook generado automaticamente con datos de 1,332 modelos de produccion.<br>
Metrica primaria: SMAPE | Desempate: MASE | Segundo desempate: RMSE<br>
Tecnologico de Monterrey -- MNA -- Proyecto Integrador TC5035.10
</small>
</center>